In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, TimestampType, BooleanType

spark = SparkSession.builder.appName("Occurence-bronze").getOrCreate()

#This get the snapshot date we will process (The oldest pending snapshot)
pending_df = spark.sql(
    """
    SELECT snapshot_date 
    FROM philippinebiodiveristyplatform.bronze.backfill_tracker 
    WHERE status = 'PENDING' 
    ORDER BY snapshot_date ASC 
    LIMIT 1
"""
)

#This checks if there are any snapshots left to process
if pending_df.count() == 0:
  print("All snapshots have been successfully processed!")
  dbutils.notebook.exit("Done")

snapshot_date_str = pending_df.collect()[0]["snapshot_date"]

df = spark.read.format("parquet").load(
    f"s3a://gbif-open-data-af-south-1/occurrence/{snapshot_date_str}/occurrence.parquet/"
)

ph_filtered = (
    df.filter(F.col("countrycode") == "PH")
    .withColumn("ingestion_timestamp", F.current_timestamp())
    .withColumn("source_file_path", F.col("_metadata.file_path"))
    .withColumn("snapshot_date", F.lit(snapshot_date_str).cast("date"))
)

# Skips the snapshot if it has already been processed
(
    ph_filtered.write.format("delta")
    .mode("overwrite")
    .option("replaceWhere", f"snapshot_date = '{snapshot_date_str}'")
    .option("mergeSchema", "true")
    .saveAsTable("philippinebiodiveristyplatform.bronze.occurrence")
)

# Marks the snapshot as completed
spark.sql(
f"""
    UPDATE philippinebiodiveristyplatform.bronze.backfill_tracker 
    SET status = 'COMPLETED' 
    WHERE snapshot_date = '{snapshot_date_str}'
"""
)

print(f"Successfully processed and marked complete: {snapshot_date_str}")